<div style="font-size:2.3em;font-weight:600;margin-bottom:10px">Agentic Artificial Intelligence</div>
<div style="font-size:1.7em;margin-bottom:14px">Unit 03 · LangGraph: state, graphs and memory</div>

Today our agent gets **steps** and a **memory**.

<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px 32px"><div><b>1</b>&nbsp; Opening: from vending machine to agent</div><div><b>6</b>&nbsp; Does a state remember?</div><div><b>2</b>&nbsp; First call, and why we use a class</div><div><b>7</b>&nbsp; Memory by hand again, and a bug to find</div><div><b>3</b>&nbsp; A system prompt, and the first surprise</div><div><b>8</b>&nbsp; The checkpointer: real memory</div><div><b>4</b>&nbsp; Memory by hand</div><div><b>9</b>&nbsp; From the notebook into our package</div><div><b>5</b>&nbsp; Graphs: state, nodes and edges</div><div><b>10</b>&nbsp; Wrap-up</div></div>

<div style="height:44px"></div>

## Part 1 of 10 · Opening: from vending machine to agent

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**Why do we want control over the steps?**

#### 1.1 · What is LangGraph?

| | Think of cooking | What it gives us |
|:--|:--|:--|
| **LangChain** | the ingredients | the parts, for example an easy way to talk to a model |
| **LangGraph** | the recipe | the order of the steps |

A model with a lot of freedom can be very creative, but then it is hard to know what it will do next.

With LangGraph, **we** decide the steps, and the model works inside those steps.

<div style="height:44px"></div>

## Part 2 of 10 · First call, and why we use a class

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How do we talk to a model, and why do we put that into a class?**

In [ ]:
# 2.1 · Imports
# The key
from dotenv import load_dotenv
load_dotenv()

# The model
from langchain.chat_models import init_chat_model

# The messages
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage

# The graph
from langgraph.graph import START, END, MessagesState, StateGraph

# The memory
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
# 2.2 · Our first call to the model
llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

response = llm.invoke([HumanMessage(content='Hi, I am Tim!')])
print(response.text)

In [ ]:
# 2.3 · A class is a box for our agent
class Agent:
    def __init__(self, llm):
        self.llm = llm

    def run(self, message: str):
        return self.llm.invoke([HumanMessage(content=message)])

agent = Agent(llm)

In [ ]:
# 2.4 · Talk to the agent
print(agent.run("Hi, I am Tim!").text)

<div style="height:44px"></div>

## Part 3 of 10 · A system prompt, and the first surprise

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What changes when the agent gets a name and instructions?**

In [ ]:
# 3.1 · An agent with a system prompt
class Agent:
    def __init__(self, llm, name: str, system_prompt: str = None):
        self.llm = llm
        self.name = name
        self.system_prompt = f"You are a helpful assistant called {self.name}."

    def run(self, message: str):
        return self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=message)])

agent = Agent(llm, "Petra")

In [ ]:
# 3.2 · I introduce myself
print(agent.run("Hi, I am Tim! What is your name?").text)

In [ ]:
# 3.3 · Does she know me?
print(agent.run("What was my name again?").text)

#### 3.4 · What did we learn?

<details>
<summary>Click to show</summary>

A language model **forgets everything after each call**. We call this **stateless**.

When a chat app seems to remember you, the app sends the old messages again. The app remembers, not the model.

> **Today's sentence:** The model remembers nothing, so we must give it everything, every time.

</details>

<div style="height:44px"></div>

## Part 4 of 10 · Memory by hand

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How can we give the agent a memory ourselves?**

In [ ]:
# 4.1 · Keep a list of all messages
class Agent:
    def __init__(self, llm):
        self.llm = llm
        self.name = "Peter"
        self.system_prompt = f"You are an assistant called {self.name} that always pranks the user."
        self.conversation = [SystemMessage(content=self.system_prompt)]

    def run(self, message: str):
        human_message = HumanMessage(content=message)
        human_message.pretty_print()
        self.conversation.append(human_message)
        response = self.llm.invoke(self.conversation)
        self.conversation.append(AIMessage(content=response.text))
        return response

agent = Agent(llm)

In [ ]:
# 4.2 · I introduce myself
print(agent.run("Hi I am Tim").text)

In [ ]:
# 4.3 · Does he know me now?
print(agent.run("What was my name again?").text)

In [ ]:
# 4.4 · Look at the list
agent.conversation

#### 4.5 · So why not stop here?

<details>
<summary>Click to show</summary>

**1.** If I restart the notebook, the list is gone.

**2.** There is only one list, so two users would share one conversation.

**3.** We do all this work by hand, and that is where bugs come from.

</details>

<div style="height:44px"></div>

## Part 5 of 10 · Graphs: state, nodes and edges

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How does LangGraph build an agent?**

#### 5.1 · A graph is points and lines

```
┌─────────┐     ┌─────────┐     ┌─────────┐
│  START  │ ──▶ │   LLM   │ ──▶ │   END   │
└─────────┘     └─────────┘     └─────────┘
```

| In a graph | In our agent | What it means |
|:--|:--|:--|
| a point | **node** | a step |
| a line | **edge** | what comes next |

In [ ]:
# 5.2 · Start the plan
workflow = StateGraph(state_schema=MessagesState)

#### 5.3 · What is a state?

The **state** is the data that flows through the agent. Think of it as the agent's “working memory”.

`MessagesState` is a ready-made state with one field, `messages`. New messages are **added** to the list. They do not replace it.

In [ ]:
# 5.4 · A node is a Python function
def llm_node(state: MessagesState) -> MessagesState:
    response = llm.invoke(state["messages"])
    return {"messages": response}

In [ ]:
# 5.5 · Put the node and the edges into the plan
workflow.add_node("llm_node", llm_node)

workflow.add_edge(START, "llm_node")
workflow.add_edge("llm_node", END)

graph = workflow.compile()

In [ ]:
# 5.6 · Run the graph
state = graph.invoke({"messages": [HumanMessage(content='Hi, I am Tim')]})

print(state["messages"][-1].text)

In [ ]:
# 5.7 · Look inside the state
state

In [ ]:
# 5.8 · Draw the graph
graph

<div style="height:44px"></div>

## Part 6 of 10 · Does a state remember?

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**We have a state now, and it keeps messages. So will the agent remember my name?**

In [ ]:
# 6.1 · Put the graph into our class
class Agent:
    def __init__(self, llm):
        self.llm = llm
        self.name = "Peter"
        self.system_prompt = f"You are a helpful assistant called {self.name}."
        self.graph = self._build_graph()

    def _build_graph(self):
        workflow = StateGraph(state_schema=MessagesState)
        workflow.add_node("llm_node", self.llm_node)
        workflow.add_edge(START, "llm_node")
        workflow.add_edge("llm_node", END)
        return workflow.compile()

    def llm_node(self, state: MessagesState) -> MessagesState:
        response = self.llm.invoke(state["messages"])
        return {"messages": response}

    def run(self, message: str):
        human_message = HumanMessage(content=message)
        human_message.pretty_print()
        state = self.graph.invoke({"messages": [
            SystemMessage(content=self.system_prompt),
            human_message]})
        return state

agent = Agent(llm)

In [ ]:
# 6.2 · I say hello
state = agent.run("Hi I am Tim")
print(state['messages'][-1].text)

In [ ]:
# 6.3 · Does he know me?
state = agent.run("What was my name again?")
print(state['messages'][-1].text)

#### 6.4 · What did we learn?

<details>
<summary>Click to show</summary>

**A state is not a memory.**

The state lives for **one run** of the graph. Each time we call `invoke`, it starts with what we give it.

</details>

<div style="height:44px"></div>

## Part 7 of 10 · Memory by hand again, and a bug to find

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**Can we simply keep the state ourselves?**

In [ ]:
# 7.1 · Keep the state in the object
class Agent:
    def __init__(self, llm):
        self.llm = llm
        self.name = "Peter"
        self.system_prompt = f"You are a helpful assistant called {self.name}."
        self.graph = self._build_graph()
        self.state = {'messages': [SystemMessage(content=self.system_prompt)]}

    def _build_graph(self):
        workflow = StateGraph(state_schema=MessagesState)
        workflow.add_node("llm_node", llm_node)
        workflow.add_edge(START, "llm_node")
        workflow.add_edge("llm_node", END)
        return workflow.compile()

    def llm_node(self, state: MessagesState) -> MessagesState:
        messages = state["messages"]
        response = self.llm.invoke(messages)
        return {"messages": [response]}

    def run(self, message: str):
        human_message = HumanMessage(content=message)
        self.state['messages'].append(human_message)
        response = self.graph.invoke(self.state)
        return response

agent = Agent(llm)

In [ ]:
# 7.2 · I introduce myself
state = agent.run("Hi I am Tim. What is your name?")
print(state['messages'][-1].text)

In [ ]:
# 7.3 · Does he know me?
state = agent.run("What was my name again?")
print(state['messages'][-1].text)

In [ ]:
# 7.4 · Count the messages
for message in state["messages"]:
    print(type(message).__name__, ":", message.text)

#### 7.5 · What did we find?

<details>
<summary>Click to show</summary>

Peter's first answer is missing. `invoke` gives us a new state, but we **never save it**.

The fix is one line, right after `invoke`:

```python
self.state = response
```

**Nothing looked wrong:** no error, no crash, and even the right answer.

</details>

<div style="height:44px"></div>

## Part 8 of 10 · The checkpointer: real memory

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How do we get real memory, with very little code?**

#### 8.1 · Checkpointer and thread

| | What it does | At the hospital |
|:--|:--|:--|
| **checkpointer** | saves the state automatically, after every step | someone keeps your folder |
| **`thread_id`** | says which conversation we are in | your patient number |

In [ ]:
# 8.2 · An agent with a checkpointer
class Agent:
    def __init__(self, llm):
        self.llm = llm
        self.name = "Peter"
        self.system_prompt = f"You are a helpful assistant called {self.name}."
        self.memory = InMemorySaver()                          # ◀ ONE: the storage
        self.graph = self._build_graph()

    def _build_graph(self):
        workflow = StateGraph(state_schema=MessagesState)
        workflow.add_node("llm_node", self.llm_node)
        workflow.add_edge(START, "llm_node")
        workflow.add_edge("llm_node", END)
        return workflow.compile(checkpointer=self.memory)      # ◀ TWO: the connection

    def llm_node(self, state: MessagesState) -> MessagesState:
        response = self.llm.invoke(self.add_system_message(state["messages"]))
        return {"messages": response}

    def run(self, message: str, thread_id: str = "default"):
        human_message = HumanMessage(content=message)
        human_message.pretty_print()
        config = {"configurable": {"thread_id": thread_id}}    # ◀ THREE: which conversation?
        snapshot = self.graph.get_state(config)
        if snapshot and snapshot.values:
            messages = snapshot.values.get("messages", [])
        else:
            messages = [SystemMessage(content=self.system_prompt)]
        messages.append(human_message)
        state = self.graph.invoke({"messages": messages}, config)
        return state["messages"][-1]

    def add_system_message(self, messages: list[BaseMessage]):
        if not any(isinstance(msg, SystemMessage) for msg in messages):
            return [SystemMessage(content=self.system_prompt)] + messages
        return messages

    def get_conversation_history(self, thread_id: str = "default"):
        config = {"configurable": {"thread_id": thread_id}}
        snapshot = self.graph.get_state(config)
        if snapshot and snapshot.values:
            return snapshot.values.get("messages", [])
        return []

agent = Agent(llm)

In [ ]:
# 8.3 · Three messages in one conversation
print(agent.run("Hi, my name is Tim and I love programming in Python!", thread_id="user1").text)

print(agent.run("What's my name and what do I love?", thread_id="user1").text)

print(agent.run("What's your name?", thread_id="user1").text)

In [ ]:
# 8.4 · Read the saved messages
history = agent.get_conversation_history(thread_id="user1")
for i, msg in enumerate(history):
    print(f"  {i+1}. {msg.__class__.__name__}: {msg.text}")

print("\n" + "="*50 + "\n")

In [ ]:
# 8.5 · A new thread: user 2
print(agent.run("What's my name?", thread_id="user2").text)

In [ ]:
# 8.6 · Back to user 1
print(agent.run("Do you still remember my name?", thread_id="user1").text)

#### 8.7 · State and memory

<details>
<summary>Click to show</summary>

**State:** the data during one run.

**Memory:** the state is saved and loaded again for the next run.

**Checkpointer:** connects the two. `InMemorySaver` keeps everything only inside the running program. A real product uses a database, like SQLite or Postgres.

*For home: the section “Key Features Added” in your own notebook has more details.*

</details>

<div style="height:44px"></div>

## Part 9 of 10 · From the notebook into our package

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How do we use all of this again in the next units?**

In [ ]:
# 9.1 · Only write what is special
from agentic_ai.agents.base import BaseAgent, State

class Agent(BaseAgent):
    def __init__(self, llm, system_prompt, checkpointer=None):
        super().__init__(llm, system_prompt=system_prompt, checkpointer=checkpointer)
        self.name = "Peter"

    def _build_graph(self):
        workflow = StateGraph(state_schema=State)
        workflow.add_node("llm_node", self.llm_node)
        workflow.add_edge(START, "llm_node")
        workflow.add_edge("llm_node", END)
        self.compile_graph(workflow)

    def llm_node(self, state: State):
        messages = self.add_system_message(state["messages"])
        response = self.llm.invoke(messages)
        return {"messages": [response]}

In [ ]:
# 9.2 · Create the agent
llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")
system_prompt = "You are a helpful assistant called Peter who talks like a pirate."
memory = InMemorySaver()
agent = Agent(llm, checkpointer=memory, system_prompt=system_prompt)

In [ ]:
# 9.3 · Talk to it
print("First message:")
print(agent.run("Hi, I am Tim!")['messages'][-1].text)

print("\nSecond message (should remember the name):")
print(agent.run("What was my name again?")['messages'][-1].text)

In [ ]:
# 9.4 · One line: SimpleAgent from our package
from agentic_ai.agents.simple_agent import SimpleAgent

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")
system_prompt = "You are a helpful assistant called Simply who talks like a star wars robot."
memory = InMemorySaver()

simple_agent = SimpleAgent(llm, name="Simply", system_prompt=system_prompt, checkpointer=memory)

In [ ]:
# 9.5 · Say hello
print(simple_agent.run("Hello, I am Luci Groundstander. What is your name?")['messages'][-1].text)

In [ ]:
# 9.6 · A joke (if we have time)
print(simple_agent.run("Please tell me a joke about Darth Vader")['messages'][-1].text)

<div style="height:44px"></div>

## Part 10 of 10 · Wrap-up

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span></div>

**What should you remember from today?**

#### 10.1 · Four things

**1.** The model remembers nothing, so we must give it everything, every time.

**2.** The **state** is the folder for one run.

**3.** **Nodes** are the steps, and **edges** say what comes next.

**4.** A **checkpointer** with a `thread_id` keeps the folder for the next run. That is real memory.

---

**At home:** find the class where Peter's answer got lost, add one line, and count the messages again.

**Next time, Unit 4:** prompts.